# 3.4 Dynamic Models (adaptado a lab local)

**Qué se ve aquí:** `@wrap_model_call` intercepta **cada llamada al modelo** y puede cambiar el modelo con `request.override(model=...)`. Aquí: conversación corta → modelo estándar; más de 10 mensajes → modelo con más contexto.

**Local:** estándar = `gemma4:latest` (8K de contexto); grande = `qwen3:8b` con `num_ctx=32768`. `response_metadata["model_name"]` muestra cuál respondió.
> En CPU, cambiar de modelo obliga a Ollama a **cargar el otro** (segundos extra en la primera llamada). En producción el criterio suele ser costo/latencia vs. capacidad.

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

### 📚 Teoría base del módulo 3: el ciclo del agente y dónde se engancha el middleware
`create_agent` es un **bucle**: el modelo piensa → (si pide tools) se ejecutan → el modelo vuelve a pensar → … hasta que responde sin pedir tools.

```
invoke ──► [before_agent] ──► ┌─► [before_model] ─► wrap_model_call( MODELO ) ─► [after_model] ─┐
                              │                                                                  │
                              │   ¿pidió tools? ── no ──► respuesta final ──► [after_agent] ──► fin
                              │        │ sí
                              └── wrap_tool_call( TOOL ) ◄┘
```
| Hook | Corre… | Ejemplo en este módulo |
|---|---|---|
| `before_agent` | 1 vez al empezar el `invoke` | borrar `ToolMessage`s (3.2) |
| `before_model` | antes de **cada** llamada al modelo | `SummarizationMiddleware` (3.2) |
| `wrap_model_call` | **envolviendo** cada llamada al modelo | cambiar modelo (3.4), tools (3.4, 3.5), prompt (`@dynamic_prompt` está hecho sobre esto) |
| `after_model` | después de cada respuesta del modelo | `HumanInTheLoopMiddleware` revisa los `tool_calls` y pausa (3.3) |
| `wrap_tool_call` | envolviendo cada ejecución de tool | reintentos o errores (como el interceptor MCP de M2.4) |

**Analogía:** el agente es un empleado; el middleware son los **supervisores** que revisan su trabajo en momentos fijos (al entrar, antes de hablar con el cliente, antes de enviar un correo…). El empleado no cambia; cambian las reglas que lo rodean.

### 📚 Teoría: modelos dinámicos (`@wrap_model_call`)
**Idea:** no todas las preguntas necesitan el mismo modelo. Uno chico y rápido para lo simple; uno más grande (o con más contexto) para lo difícil o largo. Es **enrutamiento de modelos** (*model routing*).

**`@wrap_model_call` = "envolver" la llamada al modelo.** Tu función recibe:
- `request` (`ModelRequest`): todo lo que se le va a mandar al modelo: `model`, `messages`, `system_message`, `tools`, `tool_choice`, `response_format`, `state`, `runtime`, `model_settings`.
- `handler`: la función que **de verdad** llama al modelo.

Haces cambios con **`request.override(...)`** (devuelve un request **nuevo**; no modifica el original) y llamas `handler(request)`.

**Criterios típicos para elegir modelo:** largo de la conversación (este ejemplo), tipo de usuario (gratis/pago), dificultad de la tarea, si hay imágenes, o **fallback** si el primero falla.

**En el lab local:** `gemma4` (8K de contexto, rápido) vs. `qwen3:8b` con 32K. En CPU, la primera vez que se usa el otro modelo Ollama tiene que **cargarlo** → segundos extra.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (raíz del repo) y definir el modelo local.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [1]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-3/ y local_model.py en la raíz del repo.
#         MODEL reemplaza a "gpt-5-nano" en todas las celdas (gemma4: soporta tools, el más rápido del lab).
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model, OLLAMA_URL

MODEL = get_model("gemma4:latest")

### 📘 El middleware que elige el modelo
- `len(request.messages)` cuenta los mensajes de la conversación.
- Más de 10 → `large_model`; si no, `standard_model`.
- `request.override(model=model)` cambia **solo** el modelo de esta llamada; el resto del request queda igual.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: Claude Sonnet y "gpt-5-nano" son APIs de pago → dos modelos locales de distinto tamaño y contexto.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
# from langchain.chat_models import init_chat_model
# from typing import Callable
#
# large_model = init_chat_model("claude-sonnet-4-6")
# standard_model = init_chat_model("gpt-5-nano")
#
#
# @wrap_model_call
# def state_based_model(request: ModelRequest, 
# handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
#     """Select model based on State conversation length."""
#     # request.messages is a shortcut for request.state["messages"]
#     message_count = len(request.messages)  
#
#     if message_count > 10:
#         # Long conversation - use model with larger context window
#         model = large_model
#     else:
#         # Short conversation - use efficient model
#         model = standard_model
#
#     request = request.override(model=model)  
#
#     return handler(request)

In [2]:
# 🟢 ADAPTADO LOCAL
# Motivo: Claude Sonnet y "gpt-5-nano" son APIs de pago → dos modelos locales de distinto tamaño y contexto.
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

large_model = get_model("qwen3:8b", num_ctx=32768)      # "grande": más parámetros y más contexto
standard_model = get_model("gemma4:latest")             # "estándar": el más rápido del lab


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

### 📘 El `model=` de `create_agent`
Es el modelo **por defecto**. Como el middleware lo reemplaza en **cada** llamada, aquí casi nunca se usa; igual es obligatorio pasarlo.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago. (el modelo por defecto; el middleware lo reemplaza en cada llamada)
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
#
# agent = create_agent(
#     model="gpt-5-nano",
#     middleware=[state_based_model],
#     system_prompt="You are roleplaying a real life helpful office intern."
# )

In [3]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" es API de pago. (el modelo por defecto; el middleware lo reemplaza en cada llamada)
from langchain.agents import create_agent

agent = create_agent(
    model=MODEL,
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

### 📘 Conversación corta (1 mensaje) → modelo estándar
Después, `response_metadata["model_name"]` dice **qué modelo respondió** de verdad. Debería ser `gemma4:latest`.

In [4]:
# Sin cambios
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

Oh, hi! 👋 Let me check on that for you right away.

*(I quickly look over to the office plant, maybe giving it a little gentle pat.)*

Hmm, I don't think I did today. I was swamped with filing those expense reports this morning! 😅

Would you like me to water it right now? Or is there a specific time you'd prefer I do it? I can make sure it gets a good drink! 😊


In [5]:
# Sin cambios
print(response["messages"][-1].response_metadata["model_name"])

gemma4:latest


In [6]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Did you water the office plant today?
================================== Ai Message ==================================

Oh, hi! 👋 Let me check on that for you right away.

*(I quickly look over to the office plant, maybe giving it a little gentle pat.)*

Hmm, I don't think I did today. I was swamped with filing those expense reports this morning! 😅

Would you like me to water it right now? Or is there a specific time you'd prefer I do it? I can make sure it gets a good drink! 😊


### 📘 Conversación larga (11 mensajes) → modelo grande
Ahora son más de 10 mensajes → el middleware cambia a `qwen3:8b`. Comprueba con `model_name`. Si tarda más, es la **carga** del segundo modelo en Ollama.

In [7]:
# Sin cambios
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

We should consider repotting when the roots start coming out of the drainage holes or if the plant outgrows its current pot, which might be in a few months. Keep an eye on the soil drying out faster than usual—that’s a sign it might need more space!


In [8]:
# Sin cambios
print(response["messages"][-1].response_metadata["model_name"])

qwen3:8b


In [9]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Did you water the office plant today?
================================== Ai Message ==================================

Yes, I gave it a light watering this morning.
================================ Human Message =================================

Has it grown much this week?
================================== Ai Message ==================================

It's sprouted two new leaves since Monday.
================================ Human Message =================================

Are the leaves still turning yellow on the edges?
================================== Ai Message ==================================

A little, but it's looking healthier overall.
================================ Human Message =================================

Did you remember to rotate the pot toward the window?
================================== Ai Message ==================================

I rotated it a quarter turn so it gets

## 🔎 EXTRA: ¿dónde se ve qué modelo eligió el middleware?
El middleware **no es una tool**: no aparece como `tool_call` ni como `ToolMessage`. Decide **dentro** de la llamada al modelo. Para verlo hay tres caminos:
| Dónde | Qué mirar |
|---|---|
| **La respuesta** | `response["messages"][-1].response_metadata["model_name"]` (lo que ya hicimos) |
| **LangSmith** | Abre el run → paso del modelo (`ChatOllama`) → pestaña **Metadata**: `ls_model_name` = `gemma4:latest` o `qwen3:8b`. También en *Invocation params* → `model` |
| **Un log propio** | Un `print` (o logger) dentro del middleware: deja rastro de **por qué** decidió (aquí, cuántos mensajes había) |

La celda siguiente hace la versión con log y etiqueta el run (`tags`) para encontrarlo fácil en LangSmith.

In [ ]:
# 🟢 EXTRA (aprendizaje): el mismo middleware, pero deja rastro de su decisión
@wrap_model_call
def state_based_model_log(request: ModelRequest,
                          handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    n = len(request.messages)
    model = large_model if n > 10 else standard_model
    print(f"🔀 middleware: {n} mensajes → {model.model}")
    return handler(request.override(model=model))

agent_log = create_agent(
    model=MODEL,
    middleware=[state_based_model_log],
    system_prompt="You are roleplaying a real life helpful office intern.",
)

r = agent_log.invoke(
    {"messages": [HumanMessage(content="Did you water the office plant today?")]},
    config={"tags": ["M3.4-dynamic-models"]},
)
print("Respondió:", r["messages"][-1].response_metadata["model_name"])